# 01. Khảo sát dữ liệu (EDA) - Dự báo nhu cầu điện năng

**Mục tiêu:** dự báo lượng điện (kW trung bình) của **giờ kế tiếp** từ **24 giờ gần nhất** cho một khách hàng.

Notebook này trả lời bốn câu hỏi: dữ liệu có sạch không, **chọn chuỗi nào và dựa trên tiêu chí gì**, chia tập ra sao, và có điểm bất thường nào cần ghi nhớ cho phần phân tích lỗi.

**Cách chạy:** mở notebook trong thư mục `notebooks/`, chạy lần lượt từ trên xuống. Cần có trước:
`data/processed/series_hourly.csv` (tạo bởi `python src/data_preprocessing.py`), các bảng chọn chuỗi
(`python src/select_customers.py`) và `results/tables/dst_hours.csv` (`python src/dst_check.py`).

**Quy ước dữ liệu:** giá trị là kW trung bình mỗi giờ; nhãn thời gian là *mốc kết thúc* của giờ
(mốc 02:00 gồm 01:15, 01:30, 01:45 và 02:00).

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.config import load_config, path_from, set_seed
from src.data import load_hourly_data, split_by_time, split_summary
from src.dst_check import dst_table
from src.select_customers import longest_constant_run   # dùng CHUNG một hàm với tầng C

cfg = load_config()
set_seed(cfg["project"]["seed"])

FIG_DIR = path_from(cfg, "figures_dir"); FIG_DIR.mkdir(parents=True, exist_ok=True)
TAB_DIR = path_from(cfg, "tables_dir");  TAB_DIR.mkdir(parents=True, exist_ok=True)
MAIN = cfg["customers"]["main"]

plt.rcParams.update({"figure.dpi": 100, "axes.grid": True, "grid.alpha": 0.3})
print("Chuỗi chính:", MAIN, "| Các chuỗi dùng:", cfg["customers"]["hourly_file_columns"])

## 1. Tổng quan dữ liệu theo giờ

In [ ]:
df = load_hourly_data(cfg)   # kiểm tra: không trùng, không thiếu, không âm, cách đều 1 giờ
train, val, test = split_by_time(df, cfg)
s = df[MAIN].astype(float)

print("Số giờ:", len(df), "| từ", df.index.min(), "đến", df.index.max())
print("Giá trị thiếu:", int(df.isna().sum().sum()), "| giá trị bằng 0:", int((df == 0).sum().sum()))
df.describe().round(2)

Dữ liệu có đúng 35.064 giờ (bốn năm 2011-2014), không thiếu, không trùng và không âm ở cả bốn chuỗi.
Việc kiểm tra này được tự động hóa trong `load_hourly_data`: nếu dữ liệu bị lỗi, hàm **báo lỗi và dừng**
thay vì chạy tiếp với dữ liệu sai.

Bốn chuỗi trải trên dải tải rộng, từ 88 kW (MT_234) đến 1.121 kW (MT_314) - chênh nhau gần 13 lần.
Chuỗi chính MT_162 ở mức 289 kW.

Có **4 giờ bằng 0**, tất cả thuộc chuỗi chính MT_162. Mục 9 phân tích chúng: hai giờ rơi vào ngày
chuyển giờ, hai giờ còn lại là tiêu thụ thật bằng 0. Không giá trị nào bị sửa hay loại bỏ.

## 2. Phân phối giá trị của chuỗi chính

In [ ]:
# 40 cột: với dải 0-645 kW thì mỗi cột rộng khoảng 16 kW - đủ mịn để thấy cấu trúc
# của phân phối mà không bị nhiễu răng cưa.
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(s.values, bins=40, color="tab:blue", alpha=0.85)
axes[0].set(title=f"{MAIN}: phân phối giá trị theo giờ", xlabel="kW", ylabel="Số giờ")

by_dow = [s[s.index.dayofweek == d].values for d in range(7)]
axes[1].boxplot(by_dow, tick_labels=["T2", "T3", "T4", "T5", "T6", "T7", "CN"], showfliers=False)
axes[1].set(title=f"{MAIN}: phân phối theo thứ trong tuần", ylabel="kW")

fig.tight_layout(); fig.savefig(FIG_DIR / "eda_main_distribution.png", dpi=150); plt.show()

print(s.describe().round(2).to_string())
# Mô tả phân phối bằng phân vị của CHÍNH chuỗi, không dùng ngưỡng kW đặt cứng:
# một ngưỡng cố định chỉ đúng cho một chuỗi, đổi chuỗi là vô nghĩa.
print(f"\nTrung vị {s.median():.1f} kW so với trung bình {s.mean():.1f} kW")
h, e = np.histogram(s, bins=40)
top = np.argsort(h)[::-1][:5]
print("5 khoảng giá trị dày nhất (kW):")
for i in sorted(top):
    print(f"  {e[i]:7.1f} - {e[i+1]:7.1f} : {h[i]:5d} giờ")

**Nhận xét**

Phân phối có **hai cụm tách biệt**: một cụm ở mức thấp khoảng 65-130 kW và một cụm ở mức cao khoảng
450-485 kW. Vùng ở giữa thưa hơn hẳn.

Dấu hiệu nhận ra điều này từ số liệu: trung vị 287,4 kW gần như trùng trung bình 288,6 kW, **nhưng cả hai
đều không phải giá trị điển hình** - chúng rơi vào đúng vùng thưa giữa hai cụm. Đây là đặc điểm của phân
phối hai đỉnh, khác hẳn phân phối hình chuông nơi trung vị cũng là giá trị hay gặp nhất.

Hai cụm tương ứng hai chế độ vận hành, thấy rõ ở biểu đồ hộp bên phải: thứ Ba đến thứ Sáu nằm ở mức cao,
thứ Bảy thấp hơn và Chủ nhật thấp nhất.

Hệ quả cho phần mô hình: **sai số sẽ không phân bố đều**. Những giờ chuyển giữa hai chế độ là nơi mô hình
dễ sai nhất, và đó là chỗ cần soi khi phân tích lỗi. Phân phối hai cụm cũng là lý do nhóm cân nhắc hàm mất
mát Huber thay cho MSE nếu MSE bị các giá trị đỉnh chi phối quá mức.

## 3. Quy trình chọn chuỗi: 370 chuỗi còn 4

In [ ]:
# Các bảng do src/select_customers.py tạo ra. Tiêu chí và ngưỡng nằm ở mục `selection`
# trong configs/config.yaml, không đặt cứng trong code.
for name in ["customer_selection", "customer_main_candidates"]:
    p = TAB_DIR / f"{name}.csv"
    if p.exists():
        print(f"--- {name}.csv ---")
        display(pd.read_csv(p, encoding="utf-8-sig").set_index("customer").round(3))
    else:
        print(f"Chưa có {p.name}. Chạy: python src/select_customers.py")

In [ ]:
# Tính lại đặc trưng cho 4 chuỗi đang dùng, để notebook tự kiểm chứng bảng ở trên.
# longest_constant_run import từ src.select_customers: dùng ĐÚNG hàm mà tầng C đã
# dùng để lọc, nên hai bảng không thể nói khác nhau về cùng một đại lượng.

rows = []
for c in df.columns:
    x = df[c]
    run_max, run6 = longest_constant_run(x)
    rows.append({"customer": c, "mean_kW": x.mean(), "cv(std/mean)": x.std() / x.mean(),
                 "max_kW": x.max(), "giờ_không_đổi_dài_nhất": run_max, "số_đoạn_không_đổi_>=6h": run6,
                 "autocorr_lag1": x.autocorr(1), "autocorr_lag24": x.autocorr(24),
                 "autocorr_lag168": x.autocorr(168)})

compare = pd.DataFrame(rows).set_index("customer").round(3)
compare.to_csv(TAB_DIR / "customer_comparison_hourly.csv", encoding="utf-8-sig")
compare

**Quy trình chọn chuỗi**

Bộ dữ liệu có 370 khách hàng. Nhóm lọc qua bốn tầng, mọi ngưỡng đặt trong `configs/config.yaml`
(mục `selection`) nên chạy lại cho cùng kết quả:

| Tầng | Lọc theo | Còn lại |
|---|---|---|
| A | chuỗi phải đầy đủ từ đầu 2011 (không có đoạn 0 ở đầu) | 370 → 158 |
| C | đặc trưng theo giờ: đoạn phẳng, cv, chu kỳ tuần, dịch mức tải | 158 → 122 |
| D1 | **chuỗi chính**: `ac_lag168` cao nhất trong nhóm `ac_lag24 < 0,90` | 122 → 1 |
| D2 | chuỗi phụ trải đều theo mức tải | 122 → 3 |

Tầng A loại 212 chuỗi, tất cả vì **một lý do duy nhất**: bắt đầu muộn. Theo mô tả bộ dữ liệu, đoạn 0 ở đầu
chuỗi là giai đoạn khách hàng chưa tồn tại, không phải tiêu thụ bằng 0. Nhóm loại các chuỗi này vì mốc chia
tập là ngày cố định chung cho mọi chuỗi: giữ một khách hàng bắt đầu từ 2012 thì tập train của họ ngắn hơn,
tỷ lệ 80/10/10 không còn đúng và so sánh giữa các chuỗi mất công bằng.

**Số 0 nằm GIỮA chuỗi thì KHÔNG dùng để loại.** Đó là tiêu thụ thật bằng 0 - công tơ đã hoạt động và đo
được 0 - nên là một quan sát hợp lệ. Rủi ro thật sự là chuỗi có đoạn 0 kéo dài (công tơ hỏng), và rủi ro đó
được tầng C bắt bằng `const_run_max_h` và `cv`, nơi tiêu chí có lý do rõ ràng hơn. Thực tế tầng C loại 19
chuỗi vì đoạn giá trị không đổi quá dài.

**Vì sao chọn MT_162 làm chuỗi chính**

Tiêu chí ở tầng D1 được đặt ra **trước khi chạy bất kỳ mô hình nào**: chuỗi chính phải là chuỗi mà baseline
Seasonal Naive 24 giờ *chưa* gần như tối ưu. Lý do: nếu `ac_lag24` quá cao thì "giá trị cùng giờ hôm qua"
đã bám sát chuỗi, mô hình học sâu gần như không còn gì để cải thiện và bài toán mất ý nghĩa thực nghiệm.

Kết quả: **chỉ 4 trên 122 chuỗi** thỏa điều kiện `ac_lag24 < 0,90`. Phần lớn khách hàng trong bộ dữ liệu có
chu kỳ ngày lặp gần như máy móc. MT_162 đứng đầu nhóm đó với `ac_lag24 = 0,569` và `ac_lag168 = 0,946`,
tức có chu kỳ **tuần** rất rõ nhưng chu kỳ **ngày** không lặp máy móc - đúng loại chuỗi cần cho bài toán.
MT_124 (`ac_lag24 = 0,892`) là phương án dự phòng.

**Kiểm tra độ nhạy của ngưỡng.** Sáu ngưỡng ở tầng C và D1 là lựa chọn của nhóm, không có cơ sở lý thuyết
quy định giá trị cụ thể. Nhóm chạy lại toàn bộ quy trình với 37 bộ ngưỡng, mỗi lần đổi một ngưỡng trong dải
rộng: **36 trên 37 bộ vẫn chọn MT_162**. Trường hợp duy nhất khác là khi nới `main_max_ac_lag24` lên 0,95 -
khi đó số chuỗi đủ điều kiện nhảy từ 4 lên 62 trên 122, tức tiêu chí không còn lọc nữa. Bảng đầy đủ:
`results/tables/do_nhay_nguong.csv`. Hạn chế: đây là phép thử một-lần-một-biến, không bao phủ mọi tổ hợp.

**Vai trò của từng chuỗi**

| Chuỗi | Mức tải | `ac_lag24` | Vai trò |
|---|---|---|---|
| MT_234 | 88 kW | 0,939 | kiểm tra tổng quát, quy mô nhỏ |
| MT_217 | 214 kW | 0,968 | kiểm tra tổng quát, quy mô trung bình |
| **MT_162** | **289 kW** | **0,569** | **chuỗi chính** |
| MT_314 | 1.121 kW | 0,967 | kiểm tra tổng quát, quy mô lớn |

**Lưu ý khi đọc bảng đặc trưng:** chỉ `const_run_max_h`, `cv`, `ac_lag168`, `year_drift` và
`shift_test_vs_train` được dùng làm tiêu chí lọc ở tầng C, cộng thêm `ac_lag24` ở tầng D1 và `mean_kW` ở
tầng D2. Các cột còn lại (`std_kW`, `max_kW`, `số_đoạn_không_đổi_>=6h`, `autocorr_lag1`,
`shift_val_vs_train`, `weekday_weekend_ratio`) **chỉ để tham khảo**, không tham gia vào việc chọn chuỗi.

## 4. Chuỗi chính: hình dạng theo thời gian

In [ ]:
train_end, val_end = pd.Timestamp(cfg["split"]["train_end"]), pd.Timestamp(cfg["split"]["val_end"])

fig, ax = plt.subplots(figsize=(14, 4.5))
ax.plot(s.index, s.values, lw=0.5)
ax.axvline(train_end, color="tab:red", ls="--", label="cuối train")
ax.axvline(val_end, color="tab:green", ls="--", label="cuối validation")
ax.set(title=f"{MAIN}: toàn kỳ theo giờ", xlabel="Thời gian", ylabel="kW trung bình mỗi giờ")
ax.legend()
fig.tight_layout(); fig.savefig(FIG_DIR / "eda_main_full_series.png", dpi=150); plt.show()

Chuỗi dao động trong biên độ lớn với chu kỳ đều, không có khoảng trống và không có đoạn phẳng dài
(đoạn giá trị không đổi dài nhất chỉ 1 giờ). Mức tải giảm dần qua các năm rồi nhích lên trong 2014
(xem mục 7). Hai đường đứt là mốc cắt train/validation/test.

In [ ]:
# Tuần mẫu KHÔNG chọn tùy ý. Tiêu chí: nằm trong tập TRAIN (không nhìn dữ liệu
# validation/test khi khảo sát, kể cả chỉ để vẽ minh họa), đủ 168 giờ, không chứa
# ngày chuyển giờ, không chứa giờ bằng 0, và có trung bình gần trung bình toàn
# chuỗi nhất - để tuần được vẽ là tuần đại diện, không phải tuần nhìn đẹp.
dst_days = {pd.Timestamp(d).date() for k in ("march", "october") for d in cfg["dst"][k]}
ung_vien = []
for bat_dau, nhom in s.resample("W-MON", label="left", closed="left"):
    if len(nhom) != 168 or (nhom == 0).any():
        continue
    if nhom.index.max() > train_end:
        continue
    if any(d.date() in dst_days for d in nhom.index):
        continue
    ung_vien.append((abs(nhom.mean() - s.mean()), bat_dau, nhom.mean()))

if not ung_vien:
    raise RuntimeError(
        "Không tuần nào trong tập train thỏa mọi tiêu chí. "
        "Xem lại mốc chia tập hoặc danh sách ngày chuyển giờ trong configs/config.yaml.")
ung_vien.sort()
_, TUAN_MAU, tb_tuan = ung_vien[0]
print(f"Tuần mẫu: {TUAN_MAU.date()} -> {(TUAN_MAU + pd.Timedelta(days=6)).date()}"
      f" | trung bình {tb_tuan:.1f} kW so với toàn chuỗi {s.mean():.1f} kW")

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
week = s.loc[TUAN_MAU:TUAN_MAU + pd.Timedelta(days=6, hours=23)]
axes[0].plot(week.index, week.values)
axes[0].set(title=f"{MAIN}: tuần mẫu ({TUAN_MAU.date()})", ylabel="kW")
axes[0].tick_params(axis="x", rotation=30)

wd = s.groupby(s.index.dayofweek).mean()
axes[1].bar(["T2", "T3", "T4", "T5", "T6", "T7", "CN"], wd.values)
axes[1].set(title=f"{MAIN}: trung bình theo thứ trong tuần", ylabel="kW")

fig.tight_layout(); fig.savefig(FIG_DIR / "eda_main_week_and_weekday.png", dpi=150); plt.show()
wd.round(1).rename(index=dict(enumerate(["T2", "T3", "T4", "T5", "T6", "T7", "CN"])))

**Nhận xét**

Tuần mẫu được chọn bằng quy tắc chứ không chọn tùy ý, và **chỉ lấy trong tập train** - khảo sát dữ liệu
không được nhìn vào validation hay test, kể cả để vẽ minh họa.

Theo thứ trong tuần: thứ Ba đến thứ Sáu ở mức cao và gần bằng nhau (342-346 kW), thứ Hai thấp hơn một
bậc (309 kW), thứ Bảy 208 kW và Chủ nhật thấp nhất 126 kW. Ngày thường trung bình 337 kW, cuối tuần
167 kW - chênh **2,02 lần**.

Đây là lý do **Seasonal Naive lag 168** (cùng giờ tuần trước) hợp lý hơn lag 24 cho chuỗi này: lấy giá trị
của hôm qua sẽ sai nặng ở các mốc chuyển giữa ngày thường và cuối tuần. Cũng vì vậy mà cửa sổ đầu vào
168 giờ đáng được thử ở bước thí nghiệm cải tiến.

In [ ]:
h = s.index.hour
is_weekend = s.index.dayofweek >= 5

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(range(24), s[~is_weekend].groupby(h[~is_weekend]).mean(), marker="o", label="ngày thường")
ax.plot(range(24), s[is_weekend].groupby(h[is_weekend]).mean(), marker="o", label="cuối tuần")
ax.set(title=f"{MAIN}: trung bình theo nhãn giờ trong ngày", xlabel="Nhãn giờ (mốc kết thúc)", ylabel="kW")
ax.legend()
fig.tight_layout(); fig.savefig(FIG_DIR / "eda_main_hour_profile.png", dpi=150); plt.show()

**Nhận xét**

Ngày thường có hình dạng rõ rệt:

- Cao nhất vào buổi tối và đêm, đỉnh ở nhãn 23 (487 kW), giữ mức cao qua nhãn 00-02 (404-424 kW)
- **Tụt mạnh ở nhãn 03** xuống 232 kW, chạm đáy ở nhãn 06 (160 kW)
- **Tăng nhảy bậc ở nhãn 09** từ 211 lên 387 kW - gần gấp đôi chỉ trong một giờ
- Chùng xuống quanh nhãn 11-12 (318-326 kW) rồi tăng đều suốt buổi chiều tới đỉnh buổi tối

Cuối tuần thấp và phẳng hơn nhiều: dao động 114-330 kW, và phần ban ngày gần như không có bậc nhảy ở
nhãn 09. Chênh lệch ngày thường so với cuối tuần lớn nhất vào ban ngày.

Hai mốc **nhãn 03** và **nhãn 09** là nơi giá trị đổi nhanh nhất trong ngày. Dự đoán: đó sẽ là các giờ có
sai số lớn nhất ở phần phân tích lỗi. Nhóm sẽ kiểm chứng dự đoán này bằng bảng MAE theo giờ, không kết
luận trước.

Lưu ý khi đọc hình: nhãn là *mốc kết thúc* của giờ, nên "nhãn 09" ứng với khoảng 08:15-09:00.

## 5. Tự tương quan: căn cứ chọn độ dài cửa sổ

In [ ]:
# 192 giờ = 8 ngày: đủ để thấy cả đỉnh chu kỳ ngày (lag 24) và chu kỳ tuần (lag 168),
# cộng thêm một ngày biên để thấy hành vi ngay sau đỉnh tuần.
max_lag = 192
lags = range(1, max_lag + 1)
acf = [s.autocorr(k) for k in lags]

fig, ax = plt.subplots(figsize=(13, 4))
ax.bar(list(lags), acf, width=1.0)
for k, c in [(24, "tab:red"), (48, "tab:orange"), (168, "tab:green")]:
    ax.axvline(k, color=c, ls="--", lw=1.2, label=f"lag {k}")
ax.axhline(0, color="k", lw=0.8)
ax.set(title=f"{MAIN}: tự tương quan theo độ trễ (giờ)", xlabel="Độ trễ (giờ)", ylabel="Hệ số tự tương quan")
ax.legend()
fig.tight_layout(); fig.savefig(FIG_DIR / "eda_main_acf.png", dpi=150); plt.show()

print("Các độ trễ đáng chú ý:")
for k in [1, 2, 3, 6, 12, 24, 48, 72, 120, 144, 168]:
    print(f"  lag {k:>3}: {s.autocorr(k):.3f}")

**Nhận xét**

Hình này là căn cứ trực tiếp cho ba quyết định kỹ thuật của nhóm:

1. **Tự tương quan lag 1 cao (0,848).** Giá trị giờ liền trước giải thích được phần lớn giá trị giờ kế tiếp.
   Đây là lý do baseline **Last Value** sẽ mạnh, và mô hình học sâu phải vượt được nó mới có ý nghĩa.
2. **Đỉnh ở lag 168 (0,946) cao hơn hẳn lag 24 (0,569).** Chu kỳ tuần mạnh hơn chu kỳ ngày - khớp với
   biểu đồ theo thứ ở mục 4. Vì vậy Seasonal Naive 168 được kỳ vọng tốt hơn Seasonal Naive 24.
3. **Hệ số tụt sâu ở các độ trễ giữa tuần rồi bật lên cuối tuần.** Từ 0,569 ở lag 24, hệ số rơi xuống
   khoảng 0,24-0,26 ở các lag 48-120, rồi bật lên 0,558 ở lag 144 và đạt đỉnh 0,946 ở lag 168. Cửa sổ 24
   giờ nắm được chu kỳ ngày, nhưng phải tới cửa sổ 168 giờ mô hình mới quan sát trọn một chu kỳ tuần. Đó là
   lý do thí nghiệm cải tiến so sánh ba độ dài cửa sổ 24, 48 và 168.

Khi so sánh các độ dài cửa sổ, phải đánh giá trên **cùng một tập mốc thời gian**, nếu không số mẫu khác
nhau sẽ làm kết quả không so sánh được.

## 6. So sánh hình dạng bốn chuỗi

In [ ]:
# Chuẩn hóa z-score theo từng chuỗi để so sánh HÌNH DẠNG, không so sánh độ lớn.
week_all = df.loc[TUAN_MAU:TUAN_MAU + pd.Timedelta(days=6, hours=23)]
norm = (week_all - week_all.mean()) / week_all.std()

fig, ax = plt.subplots(figsize=(14, 4.5))
for c in norm.columns:
    ax.plot(norm.index, norm[c], lw=1.4 if c == MAIN else 0.9,
            alpha=1.0 if c == MAIN else 0.6,
            label=f"{c} (chính)" if c == MAIN else c)
ax.set(title=f"Tuần mẫu {TUAN_MAU.date()}, đã chuẩn hóa z-score theo từng chuỗi",
       xlabel="Thời gian", ylabel="Độ lệch chuẩn so với trung bình tuần")
ax.tick_params(axis="x", rotation=20); ax.legend(ncol=4)
fig.tight_layout(); fig.savefig(FIG_DIR / "eda_series_comparison.png", dpi=150); plt.show()

**Nhận xét**

Chuẩn hóa z-score cho từng chuỗi để so sánh *hình dạng* thay vì độ lớn.

MT_162 (đường đậm) có chu kỳ ngày không lặp lại giống nhau giữa các ngày - đúng điều mà
`ac_lag24 = 0,569` phản ánh bằng số. Ba chuỗi phụ có `ac_lag24` khoảng 0,94-0,97, tức dạng gần như lặp
lại từng ngày với biên độ đều đặn. Với các chuỗi đó, Seasonal Naive 24 giờ gần như đã là lời giải, nên
chúng chỉ đóng vai trò kiểm tra tính tổng quát của pipeline chứ không phải nơi so tài giữa các mô hình.

Đây là bằng chứng bằng hình cho quyết định chọn chuỗi chính ở mục 3.

## 7. Mức tải theo năm và khác biệt giữa train / validation / test

In [ ]:
# Nhãn là mốc kết thúc, nên giờ 00:00 ngày 01/01/2015 thuộc về năm 2014.
year = (df.index - pd.Timedelta(hours=1)).year
by_year = df.groupby(year).mean().round(1)
by_year.index.name = "năm"
display(by_year)

by_split = pd.DataFrame({"train": train.mean(), "validation": val.mean(), "test": test.mean()}).T.round(1)
display(by_split)

# Trung bình trượt 30 ngày = 720 giờ: đủ dài để xóa hẳn chu kỳ ngày và chu kỳ tuần,
# đủ ngắn để còn thấy biến động theo mùa và xu hướng nhiều năm.
fig, ax = plt.subplots(figsize=(14, 4))
for c in df.columns:
    ax.plot(df[c].rolling(24 * 30).mean(), label=c)
ax.axvline(train_end, color="k", ls="--", lw=0.8)
ax.axvline(val_end, color="k", ls="--", lw=0.8)
ax.set(title="Trung bình trượt 30 ngày của các chuỗi", ylabel="kW")
ax.legend(ncol=4)
fig.tight_layout(); fig.savefig(FIG_DIR / "eda_rolling_mean_all.png", dpi=150); plt.show()

**Nhận xét**

- **MT_162**: trung bình theo năm giảm dần từ 321,9 (2011) xuống 261,7 (2013) rồi nhích lên 276,9 (2014).
  Chỉ số `year_drift` bằng 0,209.
- Ba chuỗi phụ cũng giảm theo cùng hướng (MT_234: 96,0 → 82,8; MT_217: 237,2 → 203,7; MT_314:
  1.280 → 1.057). Cả bốn chuỗi cùng giảm nên nhiều khả năng đây là xu hướng chung của giai đoạn, không
  phải đặc điểm riêng của một khách hàng. Đây là **giả thuyết**, nhóm không có dữ liệu để kết luận.
- Trung bình theo tập của MT_162: train 293,5 - **validation 255,1** - test 285,0. Tập validation thấp hơn
  train khoảng **13%**, trong khi test chỉ thấp hơn 3%.

**Cần ghi vào phần hạn chế của báo cáo:** tập validation rơi vào giai đoạn có mức tải thấp nhất trong cả
chuỗi. Mô hình được chọn dựa trên validation sẽ được tinh chỉnh cho một chế độ tải thấp hơn chế độ của
tập test. Đây là hệ quả tất yếu của việc chia theo thời gian - đổi lại, cách chia này mô phỏng đúng tình
huống thực tế là dùng quá khứ để dự báo tương lai, và loại bỏ rò rỉ dữ liệu.

## 8. Chia train / validation / test (80/10/10 theo thời gian)

In [ ]:
split_tbl = split_summary({"train": train, "validation": val, "test": test})
display(split_tbl.iloc[:, :5])
print("Tổng:", len(train) + len(val) + len(test), "giờ")

Mốc cắt làm tròn theo ranh giới tháng, cắt theo thời gian và **không xáo trộn**: train 27.720 giờ (79,1%),
validation 3.672 giờ (10,5%), test 3.672 giờ (10,5%); validation và test mỗi tập 153 ngày.

Số dòng từng tập được kiểm tra tự động trong `src/data.py`: khác kỳ vọng là hàm báo lỗi và dừng, thay vì
chạy tiếp với dữ liệu sai.

Hai quy ước quan trọng cho các bước sau:

1. Khi tạo cửa sổ trượt, các giá trị đầu vào của những mẫu đầu tập validation/test được lấy từ **cuối tập
   liền trước** (dữ liệu quá khứ đã quan sát, hoàn toàn hợp lệ), nhưng **nhãn dự báo phải nằm đúng trong
   tập của nó**. Nhờ vậy mọi độ dài cửa sổ đều cho ra đúng 3.672 nhãn và so sánh được công bằng.
2. Bộ chuẩn hóa chỉ được fit trên tập train, và **mỗi chuỗi dùng một scaler riêng** để còn
   `inverse_transform` đúng về thang đo gốc.

## 9. Giờ bất thường quanh ngày chuyển giờ (DST) và các giờ bằng 0

In [ ]:
dst = dst_table(df, cfg)
dst.to_csv(TAB_DIR / "dst_hours.csv", index=False, encoding="utf-8-sig")

cols = ["date", "kind", "split", "value_01h", "value_02h", "value_03h",
        "ratio_vs_neighbors", "ref_low", "ref_high", "reason"]
display(dst[dst.customer == MAIN][cols])
print("Số ngày được đánh dấu, theo chuỗi:")
print(dst.groupby("customer")["suggest_check"].sum().to_string())

print("\nCác giờ bằng 0 (mức giờ):")
ten_thu = ["T2", "T3", "T4", "T5", "T6", "T7", "CN"]
for c in df.columns:
    z = df[c][df[c] == 0]
    print(f"  {c}: {len(z)} giờ",
          [f"{x:%Y-%m-%d %H:%M} {ten_thu[x.dayofweek]}" for x in z.index] if len(z) else "")

**Cách đánh dấu bất thường**

Ngưỡng **không đặt cứng**. Với mỗi chuỗi, `src/dst_check.py` tính tỷ lệ *giờ nhãn 02:00 / trung bình giờ
01:00 và 03:00* cho cả 1.453 ngày thường, rồi lấy phân vị 1 và 99 của phân bố đó làm cận. Ngày chuyển giờ
nào rơi ra ngoài khoảng riêng của chuỗi mới bị đánh dấu.

Vì sao phân vị 1: theo định nghĩa, 2% số ngày thường cũng nằm ngoài khoảng, nên với 8 ngày chuyển giờ mỗi
chuỗi, số lần báo nhầm kỳ vọng là 8 × 0,02 = 0,16 lần - dưới 1. Với phân vị 5 thì con số này là 0,80, gần 1
nên quá lỏng. Ngưỡng vì vậy **suy ra từ số ngày có trong dữ liệu**, không phải chọn theo cảm tính.

**Kết quả cho MT_162** (khoảng riêng: 0,96 - 1,50)

| Ngày | Loại | Tập | Tỷ lệ | Đánh dấu |
|---|---|---|---|---|
| 2011-03-27 | tháng 3 | train | 1,42 | không |
| 2011-10-30 | tháng 10 | train | 1,00 | không |
| 2012-03-25 | tháng 3 | train | 0,00 | giá trị bằng 0 |
| 2012-10-28 | tháng 10 | train | 0,00 | giá trị bằng 0 |
| 2013-03-31 | tháng 3 | train | 0,26 | thấp bất thường |
| 2013-10-27 | tháng 10 | train | 1,60 | cao bất thường |
| 2014-03-30 | tháng 3 | **validation** | 0,31 | thấp bất thường |
| 2014-10-26 | tháng 10 | **test** | 1,55 | cao bất thường |

Sáu trên tám ngày được đánh dấu, và cả bốn chuỗi đều cho cùng con số 6. Hai ngày năm 2011 không có biểu
hiện bất thường ở bất kỳ chuỗi nào.

Hiện tượng khớp với mô tả bộ dữ liệu: ngày chuyển giờ tháng 3 giờ nhãn 02:00 tụt mạnh hoặc bằng 0 vì giờ
đó không tồn tại; ngày tháng 10 giá trị tăng vì hai giờ bị gộp. **Một ngoại lệ đáng ghi:** ngày
2012-10-28 là ngày tháng 10 nhưng giá trị lại bằng 0 thay vì tăng - tài liệu và dữ liệu không khớp hoàn
toàn ở mốc này.

Mốc tháng 3/2014 nằm trong tập **validation** và mốc tháng 10/2014 nằm trong tập **test**. Chúng có thể
tạo ra lỗi dự báo lớn.

**Bốn giờ bằng 0 của MT_162**

| Thời điểm | Thứ | Giải thích |
|---|---|---|
| 2012-03-25 02:00 | Chủ nhật | ngày chuyển giờ tháng 3 |
| 2012-10-28 02:00 | Chủ nhật | ngày chuyển giờ tháng 10 |
| 2013-05-06 10:00 | Thứ Hai | tiêu thụ thật bằng 0 |
| 2013-10-07 10:00 | Thứ Hai | tiêu thụ thật bằng 0 |

Hai giờ sau rơi vào **10 giờ sáng Thứ Hai** ở một khách hàng trung bình 289 kW, hai lần cách nhau 17
tháng. Nhiều khả năng là ngừng hoạt động có kế hoạch - bảo trì hoặc cắt điện. Đây là **giả thuyết**, nhóm
chưa có dữ liệu để xác nhận.

Cả bốn giờ đều nằm trong tập **train**, nên không ảnh hưởng tới phần đánh giá.

**Quyết định: giữ nguyên dữ liệu**, không sửa và không loại bỏ. Các mốc này được ghi lại trong
`results/tables/dst_hours.csv` và dùng làm **giả thuyết** giải thích lỗi ở bước phân tích lỗi - chỉ kết
luận khi có bằng chứng từ chính dữ liệu dự báo.

## 10. Kết luận và việc tiếp theo

**Quyết định về dữ liệu**

1. **Chuỗi chính: MT_162**, chọn theo quy trình bốn tầng với ngưỡng ghi trong `configs/config.yaml`.
   Là 1 trong 4 chuỗi (trên 122 chuỗi đạt tầng C) có `ac_lag24 < 0,90`, và có `ac_lag168` cao nhất trong
   nhóm đó. Dự phòng: MT_124. Kiểm tra độ nhạy: 36/37 bộ ngưỡng cho cùng kết quả.
2. **Chuỗi phụ: MT_234 (88 kW), MT_217 (214 kW), MT_314 (1.121 kW)** - kiểm tra tính tổng quát của
   pipeline ở ba quy mô tải khác nhau.
3. Dữ liệu theo giờ (kW trung bình, nhãn là mốc kết thúc), 35.064 giờ, không thiếu, không âm.
4. Chia theo thời gian: train đến 2014-03-01 00:00, validation đến 2014-08-01 00:00, phần còn lại là test
   (27.720 / 3.672 / 3.672 giờ).
5. Baseline cần có: Last Value, Moving Average, Seasonal Naive lag 24 và lag 168.

**Hạn chế cần nêu trong báo cáo**

- Mức tải giảm dần qua các năm ở cả bốn chuỗi (`year_drift` của MT_162 là 0,209).
- Tập validation rơi vào giai đoạn mức tải thấp nhất, thấp hơn train khoảng 13%, trong khi test chỉ thấp
  hơn 3%. Mô hình chọn theo validation được tinh chỉnh cho một chế độ tải khác chế độ của test.
- Các giờ bất thường quanh ngày chuyển giờ nằm trong cả validation và test.
- Tự tương quan lag 1 cao (0,848) nên baseline Last Value là đối thủ mạnh; nếu mô hình học sâu không vượt
  được, nhóm **báo cáo trung thực và phân tích nguyên nhân**, không điều chỉnh theo tập test.
- Sáu ngưỡng ở tầng C và D1 là lựa chọn của nhóm. Kiểm tra độ nhạy cho thấy kết quả không phụ thuộc vào
  chúng, nhưng phép thử là một-lần-một-biến, không bao phủ mọi tổ hợp.

**Việc tiếp theo**

Baseline và bộ hàm đánh giá (`src/baseline.py`, `src/metrics.py`, `notebooks/02_baseline.ipynb`), tạo cửa
sổ trượt dùng chung, rồi đến MLP và LSTM.

In [3]:
import pandas as pd

# 1. Đọc file dữ liệu 15 phút vừa xuất (chuyển cột timestamp sang định dạng thời gian)
df = pd.read_csv('../data/processed/series_15min.csv', parse_dates=['timestamp'])

# 2. Chuyển dữ liệu từ bảng ngang sang bảng dọc (melt) để dễ nhóm và đếm
df_melt = df.melt(id_vars=['timestamp'], var_name='Chuoi', value_name='DienNang')

# 3. Chỉ lọc ra những thời điểm có lượng điện tiêu thụ bằng 0
df_zero = df_melt[df_melt['DienNang'] == 0].copy()

# 4. Tách riêng Ngày và Giờ từ cột timestamp
df_zero['Ngay'] = df_zero['timestamp'].dt.date
df_zero['Gio'] = df_zero['timestamp'].dt.hour

# 5. Đếm số lượng giá trị 0 theo từng Chuỗi, Ngày, và Giờ
thong_ke_0 = df_zero.groupby(['Chuoi', 'Ngay', 'Gio']).size().reset_index(name='SoLan_MatDien_Hoac_KhongDung')

# Sắp xếp để xem ngày/giờ nào bị nhiều nhất
thong_ke_0 = thong_ke_0.sort_values(by=['Chuoi', 'Ngay', 'Gio'])

# Hiển thị kết quả ra màn hình notebook
display(thong_ke_0)

,Chuoi,Ngay,Gio,SoLan_MatDien_Hoac_KhongDung
0,MT_162,2012-03-25,1,4
1,MT_162,2012-03-25,2,4
2,MT_162,2012-10-28,1,4
3,MT_162,2012-10-28,2,4
4,MT_162,2013-03-31,1,4
5,MT_162,2013-05-06,9,3
6,MT_162,2013-05-06,10,1
7,MT_162,2013-10-07,9,3
8,MT_162,2013-10-07,10,3
9,MT_162,2014-03-30,1,4


In [4]:
display(df_zero['Chuoi'].value_counts())

Chuoi
MT_162    45
MT_314    18
MT_217    17
Name: count, dtype: int64